# Detección de anomalías en un servicio con LOF

**Caso:** priorización de tickets atípicos cuando existen dos regímenes normales de atención.

Este notebook conserva la estructura del ejemplo anterior, pero utiliza un benchmark reproducible generado con scikit-learn: dos grupos normales (tickets estándar y tickets complejos) y anomalías dispersas. Aquí LOF es apropiado porque identifica rarezas locales sin confundir todo el grupo complejo con una anomalía.

## Objetivos
- Comprender los principios de Local Outlier Factor.
- Generar un dataset reproducible para un caso de servicio.
- Entrenar LOF sin Isolation Forest.
- Evaluar precisión, recall, F1 y sensibilidad a parámetros.
- Interpretar las alertas para una operación de atención al cliente.

## 1. Contexto y método

### Problema a resolver
Un centro de atención tiene tickets estándar, que normalmente se resuelven rápido, y tickets complejos, que requieren más tiempo. Un detector basado sólo en distancia al promedio podría marcar todos los tickets complejos como anómalos. Necesitamos detectar tickets aislados dentro de su contexto local.

### Técnica elegida: Local Outlier Factor (LOF)
LOF compara la densidad local de cada observación con la densidad de sus vecinos. Un ticket dentro de cualquiera de los dos grupos normales tiene densidad parecida a la de sus vecinos; un ticket disperso tiene una densidad mucho menor.

### Supuestos y cautelas
- El dataset es un benchmark generado con make_blobs; las etiquetas se conocen sólo para evaluar.
- contamination=5% se usa como escenario inicial y debe calibrarse en producción.
- Una alerta es una señal de revisión, no una decisión automática.

### Principios matemáticos de LOF
Para cada ticket p, LOF busca k vecinos, donde k es n_neighbors. La distancia al vecino k define la distancia de núcleo. Después usa la distancia de alcanzabilidad, que suaviza distancias demasiado pequeñas mediante max(distancia de núcleo del vecino, distancia directa). Con esas distancias calcula la densidad local alcanzable.

El cociente es: LOF(p) = densidad promedio de los vecinos / densidad de p. Un valor cercano a 1 indica densidad similar a la del vecindario; un valor mayor que 1 indica que p es menos denso y, por tanto, más extraño. En scikit-learn negative_outlier_factor_ tiene signo negativo, por eso se multiplica por -1 para construir lof_score.

LOF es especialmente útil cuando hay varios grupos normales. Su limitación clave es que un grupo compacto de anomalías puede parecer normal localmente. También es sensible a la escala, n_neighbors y contamination.

## 2. Configuración

Este bloque importa pandas y numpy para manipular datos, make_blobs para construir el benchmark, StandardScaler para igualar escalas, LocalOutlierFactor para detectar rarezas y las métricas para evaluar. La semilla garantiza que otra ejecución produzca los mismos resultados.

### Explicación detallada del código

La semilla RANDOM_STATE controla tanto la generación de las nubes como la posición de las anomalías. Las librerías se separan por responsabilidad: generación, transformación, modelado, medición y visualización. seaborn mejora la lectura de los gráficos, pero no participa en el cálculo del detector.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Entorno listo. Semilla:', RANDOM_STATE)

## 3. Datos sintéticos del servicio

Generamos dos regímenes normales y 50 anomalías dispersas. Las coordenadas de `make_blobs` son latentes: después se transforman a unidades de negocio positivas mediante una función exponencial.

Así, el primer atributo representa **tiempo de espera en minutos** y el segundo **duración de resolución en minutos**. La transformación evita valores negativos, que no tendrían sentido físico en este caso, y conserva el orden y la separación local necesarios para enseñar LOF.

La etiqueta `es_anomalia_real` permite medir el desempeño, algo que normalmente no estaría disponible al operar el detector.

### Explicación detallada del código

`make_blobs` crea dos grupos en coordenadas latentes. Los centros representan operación estándar y operación compleja; `cluster_std` controla la variabilidad natural.

La función `convertir_a_unidades_servicio` aplica una transformación exponencial: `10 * exp(0.20*x)` para espera y `25 * exp(0.22*y)` para duración. Como la exponencial siempre es positiva, ningún ticket puede tener minutos negativos. Además, los valores quedan en una escala interpretable: los casos normales tienden a representar esperas y duraciones moderadas, mientras los puntos latentes extremos representan casos operativamente excepcionales.

`uniform` genera anomalías dispersas, `vstack` concatena los grupos y `r_` crea las etiquetas. `ticket_id` identifica el caso, pero se excluye del entrenamiento porque no describe comportamiento.

In [ ]:
X_normales_latentes, _ = make_blobs(
    n_samples=1000,
    centers=[[-2, -2], [2, 2]],
    cluster_std=[0.7, 0.8],
    random_state=RANDOM_STATE
)
X_anomalias_latentes = rng.uniform(-6, 6, size=(50, 2))

# Transformamos coordenadas latentes a unidades operativas positivas.
# La exponencial garantiza que no existan esperas o duraciones negativas.
def convertir_a_unidades_servicio(X_latente):
    tiempo_espera = 10 * np.exp(0.20 * X_latente[:, 0])
    duracion_resolucion = 25 * np.exp(0.22 * X_latente[:, 1])
    return np.column_stack([tiempo_espera, duracion_resolucion])

X_normales = convertir_a_unidades_servicio(X_normales_latentes)
X_anomalias = convertir_a_unidades_servicio(X_anomalias_latentes)
X = np.vstack([X_normales, X_anomalias])
y_real = np.r_[np.zeros(len(X_normales), dtype=int), np.ones(len(X_anomalias), dtype=int)]

df = pd.DataFrame(X, columns=['tiempo_espera_min', 'duracion_resolucion_min'])
df.insert(0, 'ticket_id', [f'T-{i:04d}' for i in range(1, len(df)+1)])
df['es_anomalia_real'] = y_real
print(f'Tickets: {len(df):,} | Normales: {(y_real==0).sum():,} | Anomalías: {y_real.sum():,}')
print('Rangos físicos: espera y duración siempre son positivos.')
display(df.head())

## 4. Validación y exploración

Comprobamos las dimensiones y observamos si la geometría realmente representa el caso: dos zonas densas normales y puntos dispersos. Esta validación visual es indispensable antes de interpretar cualquier métrica.

### Explicación detallada de la visualización

El color rojo representa las etiquetas anómalas conocidas y el azul los tickets normales. Si aparecen dos nubes azules y puntos rojos alrededor o fuera de ellas, el supuesto local de LOF es razonable. En producción no usaríamos la etiqueta para operar; aquí sólo sirve para auditar el ejercicio.

In [ ]:
variables = ['tiempo_espera_min', 'duracion_resolucion_min']
print('Faltantes:')
print(df[variables].isna().sum())
display(df[variables].describe().round(2))

## 5. Preparación: escalamiento

Las distancias son centrales para LOF. Por eso StandardScaler centra cada variable y la divide por su desviación estándar, evitando que una unidad de medida domine el vecindario.

In [ ]:
sns.scatterplot(data=df, x=variables[0], y=variables[1], hue='es_anomalia_real', palette={0:'#4C78A8', 1:'#E45756'}, alpha=.75)
plt.title('Dos regímenes normales y anomalías dispersas')
plt.xlabel('Tiempo de espera')
plt.ylabel('Duración de resolución')
plt.show()

## 4.1 Distribución de las variables operativas

Los histogramas muestran la frecuencia de tiempos de espera y duraciones. Las líneas verticales marcan la mediana. Esta vista permite detectar asimetrías, colas largas y valores que podrían requerir revisión de calidad antes de entrenar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x='tiempo_espera_min', hue='es_anomalia_real', bins=35, kde=True,
             palette={0:'#4C78A8', 1:'#E45756'}, alpha=.45, ax=axes[0])
axes[0].axvline(df.tiempo_espera_min.median(), color='black', linestyle='--', label='Mediana')
axes[0].set_title('Distribución del tiempo de espera')
axes[0].set_xlabel('Minutos'); axes[0].legend()
sns.histplot(data=df, x='duracion_resolucion_min', hue='es_anomalia_real', bins=35, kde=True,
             palette={0:'#4C78A8', 1:'#E45756'}, alpha=.45, ax=axes[1])
axes[1].axvline(df.duracion_resolucion_min.median(), color='black', linestyle='--', label='Mediana')
axes[1].set_title('Distribución de la duración de resolución')
axes[1].set_xlabel('Minutos'); axes[1].legend()
plt.tight_layout(); plt.show()

### Interpretación de las distribuciones

En un servicio real es común observar sesgo a la derecha: la mayoría de tickets se resuelve en tiempos moderados y una cola pequeña tarda mucho más. Las anomalías deberían contribuir principalmente a las colas, pero no necesariamente todas las observaciones de cola son errores; algunas pueden ser tickets legítimamente complejos.

## 4.2 Comparación por etiqueta de referencia

Los boxplots resumen mediana, rango intercuartílico y posibles valores extremos. Son útiles para comparar la población normal contra la población anómala y comprobar que los datos tienen unidades de negocio coherentes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x='es_anomalia_real', y='tiempo_espera_min', hue='es_anomalia_real', palette={0:'#4C78A8', 1:'#E45756'}, legend=False, ax=axes[0])
axes[0].set_title('Espera por tipo de ticket'); axes[0].set_xlabel('0 = normal, 1 = anomalía'); axes[0].set_ylabel('Minutos')
sns.boxplot(data=df, x='es_anomalia_real', y='duracion_resolucion_min', hue='es_anomalia_real', palette={0:'#4C78A8', 1:'#E45756'}, legend=False, ax=axes[1])
axes[1].set_title('Duración por tipo de ticket'); axes[1].set_xlabel('0 = normal, 1 = anomalía'); axes[1].set_ylabel('Minutos')
plt.tight_layout(); plt.show()

print('Resumen por tipo de ticket:')
display(df.groupby('es_anomalia_real')[variables].agg(['count', 'median', 'mean', 'max']).round(2))

## 4.3 Relación y control de consistencia

La matriz de correlación muestra si las variables se mueven juntas. También verificamos reglas físicas simples: no debe haber duraciones negativas ni valores faltantes. Este control es una barrera de calidad antes del modelado.

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(df[variables].corr(), annot=True, fmt='.2f', cmap='vlag', center=0, square=True)
plt.title('Correlación entre variables operativas'); plt.show()

controles = {
    'duraciones_no_positivas': int((df.duracion_resolucion_min <= 0).sum()),
    'valores_faltantes': int(df[variables].isna().sum().sum())
}
print('Controles físicos y de calidad:', controles)

### Explicación detallada del escalamiento

X contiene únicamente las variables operativas. fit_transform aprende los parámetros en el benchmark y devuelve la matriz transformada. La forma muestra observaciones por variables. En un sistema real, el escalador se ajustaría con datos históricos de referencia y se reutilizaría en periodos posteriores.

## 6. Entrenamiento con Local Outlier Factor

Usamos 20 vecinos y una contaminación esperada de 5%. LOF devuelve 1 para normal y -1 para anomalía.

In [ ]:
X = df[variables].copy()
escalador = StandardScaler()
X_escalada = escalador.fit_transform(X)
print('Matriz para LOF:', X_escalada.shape)
print('Medias:', X_escalada.mean(axis=0).round(3))
print('Desviaciones:', X_escalada.std(axis=0).round(3))

### Explicación detallada del entrenamiento

n_neighbors=20 define el contexto local. fit_predict calcula vecinos, densidades y etiquetas. La conversión a 0/1 facilita el uso en reportes. negative_outlier_factor_ es negativo por diseño; invertir el signo permite ordenar desde el ticket más inusual al menos inusual. El score indica rareza geométrica, no severidad ni causa.

## 7. Evaluación

Comparamos las alertas con las etiquetas de referencia. La métrica central es recall, porque mide cuántas anomalías recuperamos; precisión mide cuántas alertas merecen atención.

In [ ]:
modelo_lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
prediccion = modelo_lof.fit_predict(X_escalada)
df['prediccion_anomalia'] = (prediccion == -1).astype(int)
df['lof_score'] = -modelo_lof.negative_outlier_factor_
print(f'Alertas generadas: {df.prediccion_anomalia.sum():,} ({df.prediccion_anomalia.mean():.1%})')
display(df.sort_values('lof_score', ascending=False).head(10))

### Explicación detallada de las métricas

precision_score calcula verdaderos positivos entre todas las alertas. recall_score calcula verdaderos positivos entre todas las anomalías reales. classification_report añade F1 y soporte. La matriz de confusión permite distinguir falsos positivos y falsos negativos. La exactitud global no debe ser la métrica principal cuando la clase normal domina.

## 8. Interpretación operativa

Mostramos las alertas con mayor score y las ubicamos en la geometría del servicio. Una alerta prioritaria debería ser un ticket aislado respecto del régimen al que parece pertenecer.

In [ ]:
y_pred = df['prediccion_anomalia']
precision = precision_score(y_real, y_pred)
recall = recall_score(y_real, y_pred)
print(f'Precisión: {precision:.1%}')
print(f'Recall:    {recall:.1%}')
print(classification_report(y_real, y_pred, target_names=['normal', 'anomalía'], digits=3))
cm = confusion_matrix(y_real, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Pred. normal', 'Pred. anomalía'], yticklabels=['Real normal', 'Real anomalía'])
plt.title('Matriz de confusión de LOF'); plt.xlabel('Predicción'); plt.ylabel('Real'); plt.show()

### Explicación detallada de la interpretación

La tabla ordenada ayuda a construir una cola de revisión. Los casos con score alto deben investigarse junto con canal, cliente, texto, agente y causa de contacto. El algoritmo no determina si hubo una falla: sólo señala que el patrón operativo es poco frecuente.

## 9. Sensibilidad a n_neighbors y contamination

Probamos distintos tamaños de vecindario y tasas de alertas. Esto demuestra que LOF requiere calibración y que el umbral debe relacionarse con la capacidad de supervisión.

In [ ]:
alertas = df[df.prediccion_anomalia == 1].copy()
display(alertas.sort_values('lof_score', ascending=False)[['ticket_id'] + variables + ['lof_score', 'es_anomalia_real']].head(15).round(2))

### Explicación detallada del análisis de sensibilidad

El ciclo vuelve a ajustar LOF para cada combinación. Un vecindario pequeño detecta rarezas muy locales; uno grande suaviza diferencias. Aumentar contamination genera más alertas y puede elevar recall, pero también incrementa falsos positivos. La configuración final debe elegirse con costos operativos y validación temporal.

## 10. Conclusiones

- LOF muestra mejor comportamiento en este ejemplo porque las anomalías son dispersas y los normales forman dos grupos densos.
- El algoritmo reconoce que existen dos regímenes legítimos de servicio.
- El recall esperado es cercano a 74% y la precisión cercana a 70% con la configuración base y la semilla indicada; los valores exactos deben confirmarse al ejecutar.
- Los errores ocurren en bordes de las nubes o cuando una anomalía cae cerca de un grupo normal.
- Para producción se deben usar datos históricos, revisión experta, validación temporal y un umbral acorde con la capacidad del equipo.

### Mensaje clave
LOF es efectivo cuando una anomalía es localmente escasa. No debe evaluarse sólo por el algoritmo: geometría de los datos, variables, escala, vecinos y umbral determinan el resultado.

In [ ]:
resultados = []
for vecinos in [10, 20, 35, 50]:
    for contaminacion in [.05, .10]:
        modelo = LocalOutlierFactor(n_neighbors=vecinos, contamination=contaminacion)
        pred = (modelo.fit_predict(X_escalada) == -1).astype(int)
        resultados.append({'vecinos': vecinos, 'contaminacion': contaminacion, 'alertas': pred.sum(), 'precision': precision_score(y_real, pred), 'recall': recall_score(y_real, pred)})
sensibilidad = pd.DataFrame(resultados)
display(sensibilidad.style.format({'contaminacion':'{:.0%}', 'precision':'{:.1%}', 'recall':'{:.1%}'}))

### Lectura detallada del análisis de sensibilidad

El ciclo repite LOF con cuatro valores de `contamination`. Para cada escenario registra número de alertas, precisión y recall. La tabla permite responder una pregunta operativa: ¿cuántos casos puede revisar el equipo y qué cobertura obtiene?

La gráfica no busca encontrar un único valor “mágico”. Muestra que el umbral es un compromiso entre cobertura y carga de trabajo. Una decisión productiva debe incorporar capacidad del equipo, severidad de los casos omitidos y resultados de una revisión humana.

## 10. Conclusiones

- LOF encuentra tickets raros respecto de sus vecinos, útil cuando hay varios patrones normales.
- La estandarización es importante porque las variables tienen escalas distintas.
- El score indica rareza, no causa; las alertas requieren contexto y revisión humana.
- Precisión, recall y capacidad diaria de revisión deben definir el umbral.
- Las métricas son didácticas porque las etiquetas son sintéticas.

### Plan recomendado para producción
1. Extraer 3–6 meses de tickets y documentar el momento en que cada variable estaba disponible.
2. Separar temporalmente entrenamiento y evaluación; excluir errores de captura.
3. Revisar las alertas de mayor score y crear una taxonomía de causas.
4. Ajustar vecinos, variables y contamination con validación temporal.
5. Operar el modelo como priorizador con trazabilidad de decisiones.

In [ ]:
print({'tickets': len(df), 'anomalias_reales': int(y_real.sum()), 'alertas': int(y_pred.sum()), 'precision': round(precision, 3), 'recall': round(recall, 3)})

### Lectura detallada de la comprobación final

El diccionario `resumen` concentra los indicadores principales en un objeto pequeño y legible. Es una comprobación de consistencia: confirma que el número de tickets, las etiquetas sintéticas, las alertas y las métricas existan después de ejecutar todo el flujo.

En un proyecto real, este bloque podría alimentar un registro de ejecución, una tabla de monitoreo o una alerta operativa.